In [0]:
result = spark.sql("""create or replace table taxi_labs.pipelines.taxi_Data_silver as 
          select ROW_NUMBER() OVER (ORDER BY tpep_pickup_datetime, tpep_dropoff_datetime) as trip_id, VendorId, tpep_pickup_datetime as pickup_datetime, tpep_dropoff_datetime as dropoff_datetime, PULocationID as LocationID, total_amount from (select distinct * from taxi_labs.pipelines.bronze_taxi)
          WHERE VendorId IS NOT NULL
          AND PUlocationID BETWEEN 0 AND 265 
          AND Fare_amount IS NOT NULL
          AND tpep_pickup_datetime <= tpep_dropoff_datetime 
          AND trip_distance BETWEEN 0 AND 500
          AND total_amount BETWEEN 0 AND 1000
          AND tpep_pickup_datetime >= '2020-01-01'
          AND tpep_pickup_datetime <= current_timestamp
          """)

display(spark.sql("select * from taxi_labs.pipelines.taxi_data_silver"))



In [0]:
spark.sql("""create or replace table taxi_labs.pipelines.gold_view as select c.zone, round(avg(b.total_amount), 2) as average_cost, count(b.locationID) as total_trips
         from taxi_labs.pipelines.taxi_data_silver as b 
          left join taxi_labs.pipelines.taxizones_silver as c
          on b.locationID = c.LocationID
          group by b.locationID, c.zone
          having total_trips > 100
          order by average_cost desc
          limit 10""")

display(spark.sql("select * from taxi_labs.pipelines.gold_view"))

In [0]:
display(spark.sql("""select row_number() over (order by tpep_pickup_datetime) as trip_id, *
from (select distinct * from taxi_labs.pipelines.bronze_taxi)"""))